# 📖 Notebook 3: Write-Through & Write-Behind Patterns

In cache-aside, the **application** is responsible for managing the cache. But what if we want the cache to stay in sync automatically when data is written?

That's where **Write-Through** and **Write-Behind** come in.

## Learning Objectives

- Implement write-through caching (sync writes to both cache and DB)
- Implement write-behind caching (async batch writes to DB)
- Understand the trade-offs between consistency and performance
- Know when to use each pattern

In [ ]:
import psycopg2
import redis
import json
import time
import threading

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "caching_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

r = redis.Redis(**REDIS_CONFIG)
r.flushdb()
print("✅ Connected and Redis cleared")

## 📝 Write-Through Caching

With write-through, every write goes to **both** the cache and the database **synchronously**. The write doesn't complete until both are updated.

```
  Application             Cache (Redis)           Database (Postgres)
      │                       │                         │
      │── 1. Write data ─────▶│                         │
      │                       │── 2. Write to DB ──────▶│
      │                       │◀── 3. DB confirms ─────│
      │◀── 4. Write done ────│                         │
      │                       │                         │
  (cache and DB are always in sync)
```

**Trade-off**: Writes are slower (two writes instead of one), but reads are always fresh.

In [ ]:
def write_through_update_price(product_id: int, new_price: float):
    """
    Write-Through: update cache AND database synchronously.
    The caller waits until both are done.
    """
    cache_key = f"product:{product_id}"
    
    # Step 1: Update the database
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE products SET price = %s, updated_at = NOW() WHERE id = %s RETURNING id, name, price",
        (new_price, product_id)
    )
    row = cursor.fetchone()
    conn.commit()
    conn.close()
    
    if not row:
        return None
    
    # Step 2: Update the cache with the new data
    # We read the full product to cache the complete object
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT p.id, p.name, p.price, p.rating_avg, p.view_count,
               p.stock_quantity, c.name
        FROM products p JOIN categories c ON p.category_id = c.id
        WHERE p.id = %s
    """, (product_id,))
    row = cursor.fetchone()
    conn.close()
    
    product = {
        "id": row[0], "name": row[1], "price": float(row[2]),
        "rating": float(row[3]), "views": row[4],
        "stock": row[5], "category": row[6]
    }
    
    # Write to cache — now cache and DB are in sync
    r.set(cache_key, json.dumps(product))
    
    return product

def read_product(product_id: int) -> dict:
    """Read from cache (always fresh with write-through)."""
    cached = r.get(f"product:{product_id}")
    if cached:
        return json.loads(cached)
    return None

print("✅ Write-through functions ready")

In [ ]:
# Demo: Write-Through keeps cache and DB in sync

# Update product price using write-through
start = time.time()
product = write_through_update_price(42, 149.99)
write_time = (time.time() - start) * 1000

print(f"✏️ Write-through update: ${product['price']} ({write_time:.2f} ms)")
print()

# Now read from cache — should be immediately fresh
start = time.time()
cached = read_product(42)
read_time = (time.time() - start) * 1000

print(f"📖 Read from cache: ${cached['price']} ({read_time:.2f} ms)")
print()

# Verify DB and cache are the same
conn = get_db()
cursor = conn.cursor()
cursor.execute("SELECT price FROM products WHERE id = 42")
db_price = float(cursor.fetchone()[0])
conn.close()

print(f"🔍 Database price:  ${db_price}")
print(f"🔍 Cache price:     ${cached['price']}")
print(f"{'✅ In sync!' if db_price == cached['price'] else '❌ Out of sync!'}")
print()
print("💡 Write-through guarantees the cache is always fresh after a write.")
print(f"   But the write took {write_time:.1f}ms (DB + cache update).")

## ⚡ Write-Behind (Write-Back) Caching

Write-behind flips the priority: writes go to the cache **immediately**, and the database is updated **asynchronously** in the background.

```
  Application             Cache (Redis)           Database (Postgres)
      │                       │                         │
      │── 1. Write data ─────▶│                         │
      │◀── 2. Done (fast!) ──│                         │
      │                       │                         │
      │                       │── 3. Async flush ──────▶│ (later, in batch)
      │                       │                         │
```

**Trade-off**: Writes are very fast, but if the cache crashes before flushing, **you can lose data**.

In [ ]:
class WriteBehindCache:
    """
    Write-Behind Cache: writes go to Redis immediately,
    then a background thread flushes changes to Postgres.
    """
    
    def __init__(self, flush_interval: float = 2.0):
        self.r = redis.Redis(**REDIS_CONFIG)
        self.dirty_keys = set()       # keys that need to be flushed to DB
        self.lock = threading.Lock()
        self.flush_interval = flush_interval
        self.flush_count = 0
        self.running = True
        
        # Start background flusher thread
        self.flusher = threading.Thread(target=self._flush_loop, daemon=True)
        self.flusher.start()
    
    def write(self, product_id: int, field: str, value):
        """Write to cache immediately. DB update happens later."""
        cache_key = f"wb_product:{product_id}"
        
        # Ensure we have the full product cached first
        if not self.r.exists(cache_key):
            conn = get_db()
            cursor = conn.cursor()
            cursor.execute(
                "SELECT id, name, price, stock_quantity FROM products WHERE id = %s",
                (product_id,)
            )
            row = cursor.fetchone()
            conn.close()
            if row:
                self.r.hset(cache_key, mapping={
                    "id": str(row[0]), "name": row[1],
                    "price": str(row[2]), "stock": str(row[3])
                })
        
        # Update the cache immediately
        self.r.hset(cache_key, field, str(value))
        
        # Mark this key as dirty (needs DB sync)
        with self.lock:
            self.dirty_keys.add((product_id, cache_key))
    
    def read(self, product_id: int) -> dict:
        """Read from cache (always has latest writes)."""
        data = self.r.hgetall(f"wb_product:{product_id}")
        if data:
            return data
        return None
    
    def _flush_loop(self):
        """Background thread: flush dirty keys to the database."""
        while self.running:
            time.sleep(self.flush_interval)
            self._flush()
    
    def _flush(self):
        """Write all dirty cache entries to the database."""
        with self.lock:
            to_flush = list(self.dirty_keys)
            self.dirty_keys.clear()
        
        if not to_flush:
            return
        
        conn = get_db()
        cursor = conn.cursor()
        
        for product_id, cache_key in to_flush:
            data = self.r.hgetall(cache_key)
            if data and "price" in data and "stock" in data:
                cursor.execute(
                    "UPDATE products SET price = %s, stock_quantity = %s WHERE id = %s",
                    (data["price"], data["stock"], product_id)
                )
        
        conn.commit()
        conn.close()
        self.flush_count += len(to_flush)
        print(f"   🔄 Flushed {len(to_flush)} items to database (total: {self.flush_count})")
    
    def stop(self):
        """Stop the background flusher and do a final flush."""
        self.running = False
        self._flush()  # final flush

print("✅ WriteBehindCache class ready")

In [ ]:
# Demo: Write-Behind is fast for writes, flushes to DB in the background

cache = WriteBehindCache(flush_interval=3.0)  # flush every 3 seconds

# Rapid-fire writes — these are instant because they only hit Redis
print("⚡ Performing 10 rapid price updates (write-behind):")
print()

write_times = []
for i in range(10):
    product_id = (i % 5) + 1  # update products 1-5
    new_price = 10.00 + i
    
    start = time.time()
    cache.write(product_id, "price", new_price)
    elapsed = (time.time() - start) * 1000
    write_times.append(elapsed)
    
    print(f"   Product {product_id}: ${new_price:.2f} ({elapsed:.2f}ms)")

avg_write = sum(write_times) / len(write_times)
print(f"\n   Average write time: {avg_write:.2f}ms")
print()
print("⏳ Waiting 4 seconds for background flush...")
time.sleep(4)

# Check if DB was updated
conn = get_db()
cursor = conn.cursor()
cursor.execute("SELECT id, price FROM products WHERE id <= 5 ORDER BY id")
print("\n📊 Database values after flush:")
for row in cursor.fetchall():
    print(f"   Product {row[0]}: ${float(row[1]):.2f}")
conn.close()

cache.stop()
print("\n✅ Write-behind cache stopped")

## ⚠️ The Data Loss Risk

Write-behind's big risk: if the cache crashes before flushing, **unflushed writes are lost**.

Let's simulate this scenario.

In [ ]:
# Simulate data loss: write to cache, then "crash" before flush

cache2 = WriteBehindCache(flush_interval=10.0)  # long flush interval

# Write a price update
cache2.write(100, "price", 999.99)
print("1️⃣ Wrote price $999.99 to cache (not flushed yet)")

# Check cache — it has the new value
cached = cache2.read(100)
print(f"2️⃣ Cache says price: ${cached['price']}")

# Check DB — it still has the old value!
conn = get_db()
cursor = conn.cursor()
cursor.execute("SELECT price FROM products WHERE id = 100")
db_price = float(cursor.fetchone()[0])
conn.close()
print(f"3️⃣ Database says price: ${db_price}")

# Simulate a crash — stop without flushing
cache2.running = False  # stop the flush loop without final flush
print()
print("💥 SIMULATED CRASH — cache process dies without flushing!")
print(f"   The $999.99 update is LOST. Database still has ${db_price}.")
print()
print("💡 This is why write-behind is only safe for data you can afford to lose,")
print("   like analytics counters, view counts, or metrics.")

## 📊 Pattern Comparison

Let's compare all three patterns we've learned so far.

In [ ]:
print("📊 Cache Pattern Comparison")
print("=" * 75)
print()
print(f"{'Pattern':<18} {'Write Speed':<14} {'Read Freshness':<16} {'Data Safety':<14} {'Complexity'}")
print("-" * 75)
print(f"{'Cache-Aside':<18} {'N/A':<14} {'Stale possible':<16} {'Safe':<14} {'Simple'}")
print(f"{'Write-Through':<18} {'Slow':<14} {'Always fresh':<16} {'Safe':<14} {'Medium'}")
print(f"{'Write-Behind':<18} {'Very fast':<14} {'Always fresh':<16} {'Risk of loss':<14} {'Complex'}")
print()
print("📋 When to Use Each:")
print()
print("  Cache-Aside     → Default choice. Works for most read-heavy systems.")
print("  Write-Through   → Reads must always be fresh. Can tolerate slower writes.")
print("  Write-Behind    → Need fast writes. OK with eventual consistency & some risk.")
print("                    Examples: view counts, analytics, metrics pipelines.")
print()
print("💡 In interviews, start with cache-aside. Only mention write-through/behind")
print("   if the problem specifically requires it.")

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Redis cleared")

## 📚 Summary

### Key Takeaways

1. **Write-Through** = sync write to cache + DB. Slow writes, fresh reads, safe.
2. **Write-Behind** = write to cache, async flush to DB. Fast writes, risk of data loss.
3. **Cache-Aside** is still the default — use the others only when the problem demands it.
4. **Write-behind** is great for analytics/metrics where occasional data loss is acceptable.

### Next Up

In **Notebook 4**, we'll tackle the hardest part of caching: **invalidation and TTL strategies** — how to keep cached data fresh without breaking everything.